In [ ]:
import requests
import pandas as pd
import numpy as np
import ast
from tqdm import tqdm
from github_helper import from_github

## Finding individual votes

In [4]:
df_voting_sessions = pd.read_csv(from_github("/voting-data/voting_sessions_enriched.csv"))
print(df_voting_sessions.groupby("Period").size()) #SHOULD WE REMOVE THIS?

Period
65     294
66    1269
67    1778
68    1727
69    2019
70    1838
71    1381
dtype: int64


We choose to drop period 65 as it only has 294 voting_sessions. Keeping it would lead to potentially very volatile calculations later, which might skew the analysis excessively.  
From here we define functions to get the voting sessions we want, get all their votes, extract the data from them (since they are in json format) and save the data into csv files. Lastly we the concatenate them all, for simpler handling.
# ARE WE DROPPING PERIOD 65 OR WHAT?

In [ ]:
request_session = requests.Session()

def get_voting_sessions_in_period(df_voting_sessions, period):
    vote_ids = df_voting_sessions[df_voting_sessions['Period'] == period]['afstemning_id'].unique() #Do i need to convert to list?
    print(f"Found {len(vote_ids)} votes in period {period}")
    return vote_ids

def get_voting_session_with_votes(afstemning_id, session = request_session):
    base_url = "https://oda.ft.dk/api/"
    all_votes = []
    skip = 0

    next_link = None

    while True:
        url = f"{base_url}Afstemning({afstemning_id})/Stemme"
        # print(f"Base URL without params: {url}")
        if next_link:
            response = session.get(next_link)
        else:
            # response = session.get(url, params=params) #Use session to reuse connections and make everything run faster
            response = session.get(url)
            # print(f"This is the URL with parameters: {response.url}")
            # response = session.get(url)

        if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
            print(f"HTTP error for {afstemning_id}: ", response.status_code)
            print("Response text:", response.text)
            return None
        else:
            try:
                data = response.json()
            except ValueError:
                print("Error: Response is not valid JSON")
                print("Response text:", response.text)
                return None
        

        # print("Original_data", data)
        votes = data.get('value')
        next_link = data.get("odata.nextLink")
        # print("Next_link is ", next_link)
        # print("Value", votes)
        # votes = contained_data
        if not votes:
            # print("not votes????")
            break
        all_votes.extend(votes)
        if not next_link: #'https://oda.ft.dk/api/Afstemning(9700)/Stemme?$skip=100
            # print("No link found")
            break

        skip += 100

    return all_votes 

def get_and_save_votes_from(df_voting_sessions, voting_period):
    afstemning_ids = get_voting_sessions_in_period(df_voting_sessions, voting_period)
    all_votes_full_period = []
    sessions_w_wrong_number_votes = []
    for afstemning_id in tqdm(afstemning_ids):
        votes_in_voting_session = get_voting_session_with_votes(afstemning_id)
        if len(votes_in_voting_session) != 179:
            sessions_w_wrong_number_votes.append((voting_period, afstemning_id, len(votes_in_voting_session)))
        all_votes_full_period.extend(votes_in_voting_session)
    
    df = pd.DataFrame(all_votes_full_period)

    #Rename the columns to something usefull
    df.rename(columns={"id": "vote_id"
                       ,"typeid": "vote_typeid"
                       ,"afstemningid": "afstemning_id"
                       ,"opdateringsdato" : "vote_opdateringsdato"
                       ,"aktørid" : "aktørid"
                       }
                       , inplace= True
                       )

    df.to_csv(f"./voting-data/df_votes_p{voting_period}.csv", index = False)
    unique_actors_in_period = df['aktørid'].unique()
    print(f"Found {len(unique_actors_in_period)} unique actors in period {voting_period}")
    return df, sessions_w_wrong_number_votes

# afstemning_id = 9700
# data_from_voting_session = get_voting_session_with_votes(afstemning_id)


In [ ]:
all_sessions_with_wrong_number_votes = []
for voting_period in df_voting_sessions['Period'].unique():
    df, sessions_w_wrong_number_votes = get_and_save_votes_from(df_voting_sessions, voting_period)
    all_sessions_with_wrong_number_votes.extend(sessions_w_wrong_number_votes)
    
df_wrong = pd.DataFrame(all_sessions_with_wrong_number_votes, columns = ["Period", "afstemning_id", "n_votes"])

#Concatenate all the dataframes into one big dataframe
df = pd.DataFrame()
for period in [65, 66, 67, 68, 69, 70, 71]:
    df_votes_from_period = pd.read_csv(from_github(f"/voting-data/df_votes_p{period}.csv"))
    df = pd.concat([df, df_votes_from_period])

df.to_csv(f"./voting-data/df_votes_all_periods.csv", index = False)

Found 294 votes in period 65


100%|██████████| 294/294 [00:46<00:00,  6.35it/s]


Found 189 unique actors in period 65


,vote_id,vote_typeid,afstemning_id,aktørid,vote_opdateringsdato
0,1481475,3,5973,1886,2021-01-28T21:27:39.627
1,1481476,1,5973,1039,2018-02-16T10:35:23
2,1481477,1,5973,220,2018-02-16T10:35:23
3,1481478,1,5973,183,2018-02-16T10:35:23
4,1481479,1,5973,198,2018-02-16T10:35:23


We´ve manually checked the voting_sessions which had the wrong number of votes in a voting session, and they are all in accordance with what is available online. The errors fall into 3 categories: 1\) Between 177 and 182; We consider them representative 2) votes are 0. They will not be counted into our later calculations anyway 3) There is one observation with 144 votes. This is in accordance with what is available online. Since it's only one observation, we just leave it, and assume that it will not skew our analysis too much.

## Find information about the politician
To map votes to politicians and parties, we have to fetch who the politician was, and what party they belonged to at a certain time.  
To do this we start by getting data from "AktørAktør" endpoint which lists relations between actors. This is done for each "Aktør" which is a politician, where politician can be both the source of the relation "fraaktørid" and the target "tilaktørid" as party-politician relations are not built consistently in the API.  
We then rebuld the fetched data into a dataframe, containing aktørid, aktør, party and relation_start. For the actors which are not listed with any relations to a party, like the Minister of Digitalization, Caroline Stage Olsen (You see the irony right?), we fetch data directly from the Aktør endpoint, get the aktør-name and manually assign them to a party from public data.  
With all relations in place, we create a dataframe containing the intervals in which politicians belong to certain parties, and finally map those interval to periods with the following logic:  
 A politician should be assigned, per period, to the party they were in when the party started. If they did not have a party at start, then they should be assigned, for that period, to the first party they join in the period.  
We assign them with this logic, as we wish to map politicians on an individual level, and if we were to have multiple nodes for a politician, that might result in weird patterns in the network, as they would not have any connection to their "prior" node, even though they are voting something similar. This would indicate worse (MODULARITY?) even though it is a result of something different.  
Lastly we apply a mapping to ensure consistent naming of parties. 

In [ ]:
df = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))

In [ ]:
request_session = requests.Session()
def get_actor_relations(aktør_id, session = request_session):
    base_url = "https://oda.ft.dk/api/AktørAktør"
    all_relations = []
    for role in ['tilaktørid', 'fraaktørid']:
        params = {
            '$filter': 
                f'{role} eq {aktør_id} and rolleid eq 15'
            ,'$expand': 'TilAktør, FraAktør'
        }
        nextLink = None
        
        
        while True:
            if nextLink:
                response = session.get(nextLink)
                # print("Used nextlink")
            else:
                response = session.get(base_url, params=params)
                # print("Used built link: ", end = "")
            # print(response.url)
            if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
                print(f"HTTP error for {aktør_id}: ", response.status_code)
                print("Response text:", response.text)
                return None
            else:
                try:
                    data = response.json()
                except ValueError:
                    print("Error: Response is not valid JSON")
                    print("Response text:", response.text)
                    return None
            relations = data.get('value')
            all_relations.extend(relations)
            nextLink = data.get('odata.nextLink')
            if not nextLink:
                break

    # return relations
    return all_relations

def build_parties_from_relations(relations_for_actor, aktør_id):
    rows = []

    for rel in relations_for_actor:
        fra_id = rel.get('fraaktørid')
        til_id = rel.get('tilaktørid')

        fra = rel.get('FraAktør') or {}
        til = rel.get('TilAktør') or {}
        startdato = rel.get('startdato')

        if fra_id == aktør_id and til.get('typeid') == 4: #Needs to be type4
            actor_id = fra.get('id')
            actor_name = fra.get('navn')
            party_name = til.get('navn')
            rows.append([actor_id, actor_name, party_name, startdato])

        elif til_id == aktør_id and fra.get('typeid') == 4:
            actor_id = til.get('id')
            actor_name = til.get('navn')
            party_name = fra.get('navn')
            rows.append([actor_id, actor_name, party_name, startdato])

    relations_df = pd.DataFrame(
        rows,
        columns=["aktørid", "aktør", "party", "relation_start"]
    )

    return relations_df

def get_actor_df_with_no_relations(aktør_id, session = request_session):
    url = f"https://oda.ft.dk/api/Aktør({aktør_id})"
    response = session.get(url)

    # print(response.url)
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
    
    kendt_aktør_id = aktør_id
    kendt_aktør_navn = data.get('navn')
    party_name = "N/A"
    relation_start = pd.Timestamp('2000-01-01T12').normalize()
    # relation_start = "2000-01-01T00:00:00"
    dict_info = {"aktørid" : [kendt_aktør_id], "aktør" : [kendt_aktør_navn], "party" : [party_name] , "relation_start" : [relation_start]}
    relations_df = pd.DataFrame(data = dict_info)

    return relations_df, kendt_aktør_navn, kendt_aktør_id

In [ ]:
party_naming_rules = [
    # Big Danish parties
    (r'^Enhedslisten',                              'Enhedslisten'),
    (r'^Socialdemokratiet$',                        'Socialdemokratiet'),
    (r'^Socialistisk Folkeparti$',                  'Socialistisk Folkeparti'),
    (r'^Dansk Folkeparti$',                         'Dansk Folkeparti'),
    (r'^Venstre, Danmarks Liberale Parti$',         'Venstre'),
    (r'^Det Radikale Venstre$',                     'Radikale Venstre'),
    (r'^Radikale Venstre$',                         'Radikale Venstre'),
    (r'^Det Konservative Folkeparti$',              'Det Konservative Folkeparti'),
    (r'^Liberal Alliance$',                         'Liberal Alliance'),
    (r'^Moderaterne$',                              'Moderaterne'),
    (r'^Ny Alliance$',                              'Liberal Alliance'),
    (r'^Alternativet$',                             'Alternativet'),
    (r'^Frie Grønne, Danmarks Nye Venstrefløjsparti$', 'Frie Grønne'),
    (r'^Venstresocialisterne$',                     'Venstresocialisterne'),

    # "Uden for folketingsgrupperne - <navn>"
    (r'^Uden for folketingsgrupperne\b',            'Uden for Folketingsgrupperne'),

    # Danmarksdemokraterne variants (dash vs en dash)
    (r'^Danmarksdemokraterne\b',                    'Danmarksdemokraterne'),

    # Kristendemokraterne / Kristeligt Folkeparti (same party, renamed)
    # If you prefer to keep them separate, split these into two different canonicals.
    (r'^Kristendemokraterne$',                      'Kristendemokraterne'),
    (r'^Kristeligt Folkeparti$',                    'Kristendemokraterne'),

    # Other Danish / Danish-rooted parties
    (r'^Fremskridtspartiet$',                       'Fremskridtspartiet'),
    (r'^Frihed 2000$',                              'Frihed 2000'),
    (r'^Borgernes Parti\b',                         'Borgernes Parti'),
    (r'^Nye Borgerlige$',                           'Nye Borgerlige'),

    # Greenlandic parties
    (r'^Inuit Ataqatigiit$',                        'Inuit Ataqatigiit'),
    (r'^Siumut$',                                   'Siumut'),
    (r'^Nunatta Qitornai$',                         'Nunatta Qitornai'),
    (r'^Naleraq$',                                  'Naleraq'),

    # Faroese parties
    (r'^Sambandsflokkurin$',                        'Sambandsflokkurin'),
    (r'^Javnaðarflokkurin$',                        'Javnaðarflokkurin'),
    (r'^Tjóðveldisflokkurin$',                      'Tjóðveldi'),
    (r'^Tjóðveldi$',                                'Tjóðveldi'),
]

def normalize_party(name: str) -> str:
    """Map raw party string to canonical party name."""
    if pd.isna(name):
        return name

    for pattern, canonical in party_naming_rules:
        if re.match(pattern, name):
            return canonical

    # If nothing matches, just return original
    return name

def make_party_intervals_from_all_actors(relations_df_for_all):
    # where_has_relation_start = (~relations_df_for_all['relation_start'].isnull())
    df_rel = relations_df_for_all.copy()
    df_rel['relation_start'] = pd.to_datetime(df_rel['relation_start'])

    # True if this aktørid has at least one non-null relation_start
    has_any_start = df_rel.groupby('aktørid')['relation_start'].transform('count') > 0

    # Rows belonging to aktørid where *all* relation_start are NaT
    mask_no_start_actor = ~has_any_start

    # First row per aktørid (based on current ordering)
    first_row_per_actor = ~df_rel.duplicated(subset='aktørid', keep='first')

    # Default start date
    default_start = pd.Timestamp('2000-01-01T12').normalize()

    # For actors with no start dates at all, set first row's relation_start
    df_rel.loc[mask_no_start_actor & first_row_per_actor, 'relation_start'] = default_start

    # where_has_relation_start_mask = (~relations_df_for_all['relation_start'].isnull())
    where_has_relation_start = df_rel['relation_start'].notna()
    df_rel = df_rel[where_has_relation_start].copy()

    # Sort so "next" makes sense (per actor)
    df_rel = df_rel.sort_values(['aktørid', 'relation_start'])

    # Next relation_start per actor
    df_rel['relation_end'] = (
        df_rel
        .groupby('aktørid')['relation_start']
        .shift(-1)
        - pd.Timedelta(days=1)
    )

    # 1) For each aktørid, detect when the party changes vs previous row
    party_change = (
        df_rel['party']
        != df_rel.groupby('aktørid')['party'].shift()
    )

    # 2) Use cumulative sum to create a "block" id of consecutive same-party rows
    df_rel['block'] = party_change.groupby(df_rel['aktørid']).cumsum()

    # 3) Group by aktørid + party + block and aggregate start/end
    collapsed = (
        df_rel
        .groupby(['aktørid', 'aktør', 'party', 'block'], as_index=False)
        .agg(
            relation_start=('relation_start', 'min'),
            relation_end=('relation_end', 'max')
        )
    )
    where_no_end = (collapsed['relation_end'].isnull())
    collapsed.loc[where_no_end, 'relation_end'] = pd.Timestamp.today().normalize()
    collapsed.drop(columns = "block", inplace = True)
    collapsed = collapsed.sort_values(['aktørid', 'relation_start'])

    collapsed['party_clean'] = collapsed['party'].apply(normalize_party)
    old_parties = collapsed['party'].unique()
    new_parties = collapsed['party_clean'].unique()
    print(f"{len(old_parties)-len(new_parties)} 'parties' removed in cleaning, where some are like 'Uden for folketingsgrupperne - <SomeName>'. The mapping can be found in 'party_naming_rules'.")
    collapsed.drop(columns = "party", inplace=True)
    collapsed.rename(columns = {"party_clean" : "party"}, inplace=True)
    
    return collapsed

def generate_party_per_period_per_actor(party_intervals):
    from folketingsperioder import periods_df

    # ----- 1) Actor × Period Cartesian product -----
    actors = party_intervals[['aktørid', 'aktør']].drop_duplicates()
    periods_subset = periods_df[
        periods_df['Period'].between(65, 71)
    ][["Period", "period_start_date"]].copy()

    actors['key'] = 1
    periods_subset['key'] = 1
    actor_periods = actors.merge(periods_subset, on='key').drop(columns='key')

    # Clean inputs
    ap = actor_periods.dropna(subset=['aktørid', 'period_start_date']).copy()
    intervals = party_intervals.dropna(subset=['aktørid', 'relation_start']).copy()

    # Sort globally on the merge key (IMPORTANT for merge_asof)
    ap_sorted = ap.sort_values('period_start_date').reset_index(drop=True)
    intervals_sorted = intervals.sort_values('relation_start').reset_index(drop=True)
    intervals_sorted = intervals_sorted.drop(columns=['aktør'])

    # ----- 2) BACKWARD merge: party before period start -----
    backward = pd.merge_asof(
        ap_sorted,
        intervals_sorted,
        left_on='period_start_date',
        right_on='relation_start',
        by='aktørid',
        direction='backward',
        allow_exact_matches=True
    )
    backward.rename(columns={'party': 'party_back'}, inplace=True)

    # ----- 3) FORWARD merge: party after period start (mid-period joiners) -----
    forward = pd.merge_asof(
        ap_sorted,
        intervals_sorted,
        left_on='period_start_date',
        right_on='relation_start',
        by='aktørid',
        direction='forward',   # relation_start >= period_start_date
        allow_exact_matches=True
    )
    forward.rename(columns={'party': 'party_fwd'}, inplace=True)

    # Combine backward + forward
    merged = backward.copy()
    merged['party_fwd'] = forward['party_fwd']

    # ----- 4) Final party logic -----
    # 1) If there was a party BEFORE the period → use that
    # 2) Otherwise if party starts INSIDE the period → use that
    merged['party'] = merged['party_back'].fillna(merged['party_fwd'])

    # If still NA → actor has no party in that period → drop
    merged = merged.dropna(subset=['party'])

    result = merged[['aktørid', 'aktør', 'Period', 'party']].reset_index(drop=True)
    return result


In [ ]:
unique_actors = df['aktørid'].unique()
all_relations_df = pd.DataFrame()
for aktørid in tqdm(unique_actors):
    relations_json = get_actor_relations(aktørid)
    if relations_json:
        relations_df = build_parties_from_relations(relations_json, aktør_id=aktørid)
    else:
        relations_df, politician_name, aktør_id = get_actor_df_with_no_relations(aktørid)
        print(f"Found no party relations in AktørAktør for {politician_name} with aktørid {aktør_id}, created it from Aktør API instead")
    all_relations_df = pd.concat([all_relations_df, relations_df])

print(f"Created party relations for {len(all_relations_df['aktørid'].unique())} unique actors out of {df['aktørid'].nunique()} unique in df_votes.")

#There are some politicians which are not correctly mapped in the system. We manually map them.
map_actor_to_party ={
    20976: "Moderaterne" #Caroline Stage Olsen. We can assign this because, as we see later, we work mainly with period 66 and 71. Caroline was not active in 66, and is in moderaterne in 71
    ,5905: "Socialdemokratiet" #Mogens Jensen
    ,3042: "Socialdemokratiet" #Frode Sørensen
    ,7633: "Socialdemokratiet" #Torben Hansen 
    ,5593: "Socialdemokratiet" #Carsten Hansen
    ,8319: "Socialdemokratiet" #Jytte Andersen
    }

for aktørid, party in map_actor_to_party.items():
    actor_observation_mask = (all_relations_df['aktørid']==aktørid)
    all_relations_df.loc[actor_observation_mask, "party"] = party

#now create party intervals for all politicians
party_intervals = make_party_intervals_from_all_actors(all_relations_df)
#based on the party interval, assign them to the party they were in at the start of the legislative period, for that period, so if they changed party, we do not consider it
party_per_period_df = generate_party_per_period_per_actor(party_intervals)

# party_intervals
party_per_period_df.to_csv("./actor-data/party_per_period.csv", index= False)

  4%|▍         | 30/710 [00:10<03:37,  3.13it/s]

Found no party relations in AktørAktør for Jytte Andersen with aktørid 8319, created it from Aktør API instead


 12%|█▏        | 83/710 [00:24<01:44,  5.99it/s]

Found no party relations in AktørAktør for Carsten Hansen with aktørid 5593, created it from Aktør API instead


 12%|█▏        | 88/710 [00:26<01:58,  5.24it/s]

...

Found no party relations in AktørAktør for Mogens Jensen, Brøndby with aktørid 5905, created it from Aktør API instead


 99%|█████████▊| 701/710 [03:35<00:01,  8.81it/s]

Found no party relations in AktørAktør for Caroline Stage Olsen with aktørid 20976, created it from Aktør API instead


100%|██████████| 710/710 [03:36<00:00,  3.27it/s]


Created party relations for 710 unique actors out of 710 unique in df_votes.
52 'parties' removed in cleaning, where some are like 'Uden for folketingsgrupperne - <SomeName>'. The mapping can be found in 'party_naming_rules'.


## Merge the information
Now we merge all the acquired data into some big dataframes we can do calculations on. We join votes onto voting sessions, and then join actors with their parties onto the merged dataframe. We also limit what columns we are keeping, so we only maintain the most needed information for the specific tasks.  

In [ ]:
df_votes = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))
df_voting_sessions = pd.read_csv(from_github("/voting-data/voting_sessions_enriched.csv"))
party_per_period = pd.read_csv(from_github("/actor-data/party_per_period.csv"))

In [ ]:
relevant_topics_for_joining_on_votes = ["afstemning_id", "afstemning_nummer", "afstemning_vedtaget", "Period", "primary_topic", "all_topics", "møde_dato", 'møde_year_month'] #The last one is a homemade one ya know
df_voting_sessions['møde_dato'] = pd.to_datetime(df_voting_sessions['møde_dato'])
df_voting_sessions['møde_year_month'] = df_voting_sessions['møde_dato'].dt.to_period('M')

#Join the voting session with the votes
votes_enriched = df_votes.merge(df_voting_sessions[relevant_topics_for_joining_on_votes], how = "left", on = "afstemning_id")

#Join the votes with the actor information
votes_with_party = votes_enriched.merge(party_per_period, how = "left", on = ["aktørid", "Period"])

#Do a little bit of cleaning
columns_to_drop = ["vote_opdateringsdato"]
df_limited = votes_with_party.drop(columns = columns_to_drop)
df_renamed = df_limited.rename(columns = {"aktør" : "politician"})



#Sanity check
print(df_renamed.groupby("Period")["afstemning_id"].nunique())
print(df_renamed.groupby("Period")["aktørid"].nunique())

Period
65     294
66    1269
67    1778
68    1727
69    2017
70    1688
71    1306
Name: afstemning_id, dtype: int64
Period
65    189
66    216
67    241
68    228
69    237
70    219
71    235
Name: aktørid, dtype: int64


In [ ]:
#And save the new data. We save it in seperate files, as the files would otherwise be too big.
for period in [65, 66, 67, 68, 69, 70, 71]:
    votes_p = df_renamed[df_renamed["Period"] == period]
    votes_p.to_csv(f"./voting-data/votes_enriched_p{period}.csv", index = False)

# Calculate edge-weights

In the following notebook, we create the edges that we will later use to build and analyze our networks. The edges are defined as follows:  
For two nodes, where nodes can be either parties or politicians, we calculate the edge-weight called "agreement", as the number of times they voted the same, out of the total amount of votes in which they both voted.  
We do not count "absent" as a vote, so the 3 types of votes are "Agree", "Disagree" and "Abstain". 
We calculate edges across all topics, as well as per individual topic, and per period. This calculation is done on both a party and a politician level.  


In [ ]:
def build_edges_for_period(filtered_df, period, period_col = "møde_year_month"):
    total_df_for_period = filtered_df[filtered_df[period_col] == period]
    df_period = total_df_for_period[['afstemning_id', 'vote_typeid', 'politician','party', period_col]] #, topic_col

    # #Now we find all pairs by joining the dataframe onto itself based on the 2 criteria.       
    pairs = df_period.merge(
        df_period
        , on = ["afstemning_id", period_col] # We do not group by the kind of vote they did, as we would rather do calculations on it seperately than create 3 df for each
        , suffixes = ("_source", "_target")
    )

    #Now we make sure we only have one observation per pair
    pairs = pairs[pairs['politician_source'] < pairs['politician_target']] #Only keep the ones where the politicians are different
    # 1) No duplicate pairs, so values are always different 
    # 2) No "reverse" pairs, as one has to be bigger than the other. If we had used != then there might have been (A,B) and (B,A)

    #Now we do something similar to ensure that we group the politician parties
    mask = pairs['party_source'] <= pairs['party_target']
    pairs['party_a'] = np.where(mask, pairs['party_source'], pairs['party_target'])
    pairs['party_b'] = np.where(mask, pairs['party_target'], pairs['party_source'])

    # #Now we have to count them.
    total_votes_shared_by_po = (
        pairs.groupby(["politician_source", "politician_target", "party_source", "party_target", period_col])
            .size() #Get the number of total votes shared 
            .rename("total_votes_shared")
    )

    total_votes_shared_by_pa = (
        pairs.groupby(['party_a', 'party_b', period_col])
        .size()
        .rename('total_votes_shared')
    )

    agreeing_pairs = pairs[pairs["vote_typeid_source"]==pairs["vote_typeid_target"]]
    agreed_votes_po = (
        agreeing_pairs.groupby(["politician_source", "politician_target"
                , "party_source", "party_target"
                , period_col]
            )
            .size()  
            .rename("total_votes_agreed")
    )

    agreed_votes_pa = (
        agreeing_pairs.groupby(["party_a", "party_b", period_col])
            .size()  
            .rename("total_votes_agreed")
    )

    #Calculate for the politician
    result_po = (
        total_votes_shared_by_po.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_po, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    result_po['weight'] = result_po["total_votes_agreed"] / result_po["total_votes_shared"]

    #Calculate for the party
    result_pa = (
        total_votes_shared_by_pa.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_pa, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index()
    )
    result_pa['weight'] = result_pa["total_votes_agreed"] / result_pa["total_votes_shared"]

    return result_po, result_pa

def build_and_save_edges_for_multiple_periods(df, topic, topic_col = "all_topics", period_col = "møde_year_month"):
    filtered_df = df[df['vote_typeid'] != 3]
    if topic != "general":
        df_exp = filtered_df.explode(topic_col) #Explode if we are calculating it for a specific topic
        filtered_df = df_exp[df_exp[topic_col] == topic] #Limit the dataframe to only the chosen topic

    all_periods = filtered_df[period_col].unique()
    all_results_for_politicians = []
    all_results_for_parties = []
    print(f"Creating dataframe for {topic}, by {period_col}", end = " ")
    for period in all_periods:
        result_po, result_pa = build_edges_for_period(filtered_df, period, period_col)
        all_results_for_politicians.append(result_po)
        all_results_for_parties.append(result_pa)

    all_votes_po = pd.concat(all_results_for_politicians, ignore_index = True)
    all_votes_pa = pd.concat(all_results_for_parties, ignore_index = True)

    #Rename for using correct names
    all_votes_po.rename(
        columns = {
            'politician_source' : 'source'
            ,'politician_target' : 'target'
            ,'party_source' : 'source_party'
            ,'party_target' : 'target_party'
        }
        , inplace = True
    )

    all_votes_pa.rename(
        columns = {
            'party_a' : 'source'
            ,'party_b' : 'target'
        }
        , inplace= True
    )

    #Add the topics
    all_votes_po['topic'] = topic
    all_votes_pa['topic'] = topic

    all_votes_po.to_csv(f"./edges/politician/{period_col}/politician_edges_{topic}_by_{period_col}.csv", index = False)
    all_votes_pa.to_csv(f"./edges/party/{period_col}/party_edges_{topic}_by_{period_col}.csv", index = False)


In [ ]:
all_topics = df_votes.explode("all_topics")['all_topics'].dropna().unique() #Drop na, because it would otherwise return an "na" column, which we don't want
topics_for_loop = [topic for topic in all_topics]
topics_for_loop.append("general") 

for topic in tqdm(topics_for_loop):
    build_and_save_edges_for_multiple_periods(df_votes, topic = topic, topic_col="all_topics", period_col = 'Period')


########### Gorup it into just one big dataframe for the parties in periods
all_party_edges = pd.DataFrame()
for topic in topics_for_loop:
    party_df = pd.read_csv(f"./edges/party/Period/party_edges_{topic}_by_Period.csv")
    all_party_edges = pd.concat([all_party_edges, party_df])

all_party_edges.to_csv(f"./edges/party/all_party_edges_by_Period.csv", index = False)

  0%|          | 0/15 [00:00<?, ?it/s]

Creating dataframe for finans_budget, by Period 

  7%|▋         | 1/15 [00:48<11:23, 48.85s/it]

Creating dataframe for klima_miljø, by Period 

 13%|█▎        | 2/15 [01:12<07:21, 33.96s/it]

...

 87%|████████▋ | 13/15 [10:46<00:58, 29.41s/it]

Creating dataframe for bolig, by Period 

 93%|█████████▎| 14/15 [11:01<00:24, 24.90s/it]

Creating dataframe for general, by Period 

100%|██████████| 15/15 [13:00<00:00, 52.02s/it]


## Calculate voting percentages
The purpose of calculating voting percentages, is to see what parties have votes for different types of topics/periods etc.. We use this later to investigate what kind of legislation invokes certain types of votings across all of the available dimensions. Voting percentages is what a party has voted on each of the individual voting sessions. Here we include "absent" votes. 

In [ ]:
def get_voting_percentages_for_party_on_voting_session(df):
    df_cast_votes = df[df["vote_typeid"] != 3]
    df_type_1 = df[df["vote_typeid"] == 1]
    df_type_2 = df[df["vote_typeid"] == 2]
    df_type_3 = df[df["vote_typeid"] == 3]
    df_type_4 = df[df["vote_typeid"] == 4]

    cols_to_group_by = ["afstemning_id", "party", "Period", "afstemning_vedtaget", "møde_year_month"]

    total_votes = (df.groupby(cols_to_group_by).size().rename("total_potential_votes"))
    cast_votes = (df_cast_votes.groupby(cols_to_group_by).size().rename("total_cast_votes"))

    votes_t1 = (df_type_1.groupby(cols_to_group_by).size().rename("votes_type_1"))
    votes_t2 = (df_type_2.groupby(cols_to_group_by).size().rename("votes_type_2"))
    votes_t3 = (df_type_3.groupby(cols_to_group_by).size().rename("votes_type_3"))
    votes_t4 = (df_type_4.groupby(cols_to_group_by).size().rename("votes_type_4"))

    voting_percentages = (
        total_votes.to_frame() #Create df for total_votes_shared
        .join([cast_votes, votes_t1, votes_t2, votes_t3, votes_t4], how = "left") #Left join as there are some people who shared votes but did not agree

        # .join(df_type_2, how = "left", on = ["voting_id", "party", "Period", "vedtaget", "Møde.dato"])
        .fillna(0)
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    voting_percentages["agree_percentage_of_cast_votes"] = voting_percentages["votes_type_1"] / voting_percentages["total_cast_votes"]
    voting_percentages["disagree_percentage_of_cast_votes"] = voting_percentages["votes_type_2"] / voting_percentages["total_cast_votes"]
    voting_percentages["abstain_percentage_of_cast_votes"] = voting_percentages["votes_type_4"] / voting_percentages["total_cast_votes"]
    voting_percentages["absent_percentage_of_potential_votes"] = voting_percentages["votes_type_3"] / voting_percentages["total_potential_votes"]

    voting_percentages.fillna(0, inplace = True)
    voting_percentages.to_csv("./voting-data/voting_session_party_percentages.csv", index = False)
    return voting_percentages

voting_per = get_voting_percentages_for_party_on_voting_session(df_votes)